In [ ]:
import os
import pretty_midi
from collections import defaultdict
from music21 import chord
import torch
import pickle
import torch.nn as nn
import torch.nn.functional as F
from music21 import stream, note
from torchinfo import summary
import re
import difflib
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split


In [ ]:
PRETRAIN_DIR = "PRE_TRAIN"
TRAIN_DIR = "MIDI_TRAIN"
TOLERANCE = 0.1
MIN_NOTES = 2
BATCH_SIZE = 8

def midi_to_chord_progression_pretrain(path):
    midi_data = pretty_midi.PrettyMIDI(path)
    chords = []
    for instrument in midi_data.instruments:
        if instrument.is_drum:
            continue
        notes = sorted(instrument.notes, key=lambda n: n.start)
        current_chord = []
        current_time = notes[0].start if notes else 0.0
        for note in notes:
            if note.start - current_time > 0.5:
                if current_chord:
                    chord_mod12 = sorted(list(set([n % 12 for n in current_chord])))
                    chords.append(chord_mod12)
                current_chord = []
                current_time = note.start
            current_chord.append(note.pitch)
        if current_chord:
            chord_mod12 = sorted(list(set([n % 12 for n in current_chord])))
            chords.append(chord_mod12)
    return chords

def midi_to_chords_train(midi_path, tolerance=TOLERANCE, min_notes=MIN_NOTES):
    pm = pretty_midi.PrettyMIDI(midi_path)
    all_notes = []
    for instrument in pm.instruments:
        if instrument.is_drum:
            continue
        all_notes.extend(instrument.notes)

    notes_by_time = defaultdict(list)
    for note in all_notes:
        time_key = round(note.start / tolerance) * tolerance
        notes_by_time[time_key].append(note.pitch)

    chords = []
    for time in sorted(notes_by_time.keys()):
        note_group = notes_by_time[time]
        if len(note_group) < min_notes:
            continue
        try:
            m21_chord = chord.Chord(note_group)
            pc_set = sorted(set(n % 12 for n in note_group))
        except Exception:
            pc_set = sorted(set(n % 12 for n in note_group))
        chords.append(pc_set)
    return chords

def transpose_progression(prog, semitones):
    return [[(n + semitones) % 12 for n in chord] for chord in prog]

def chord_to_token(chord):
    return '_'.join(map(str, sorted(chord)))

def build_vocab(progs):
    token_to_id = {'<pad>': 0, '<sos>': 1, '<eos>': 2}
    id_to_token = {0: '<pad>', 1: '<sos>', 2: '<eos>'}
    current_id = 3
    for prog in progs:
        for chord in prog:
            tok = chord_to_token(chord)
            if tok not in token_to_id:
                token_to_id[tok] = current_id
                id_to_token[current_id] = tok
                current_id += 1
    return token_to_id, id_to_token

def progression_to_token_seq(prog, token_to_id):
    return [token_to_id[chord_to_token(ch)] for ch in prog]



X_pretrain_raw = []
for file in os.listdir(PRETRAIN_DIR):
    path = os.path.join(PRETRAIN_DIR, file)
    try:
        base_prog = midi_to_chord_progression_pretrain(path)
        for shift in range(12):
            X_pretrain_raw.append(transpose_progression(base_prog, shift))
    except Exception as e:
        print(f"Error: {path}: {e}")


token_to_id_pretrain, id_to_token_pretrain = build_vocab(X_pretrain_raw)

X_pretrain = []
Y_pretrain = []
for prog in X_pretrain_raw:
    token_seq = progression_to_token_seq(prog, token_to_id_pretrain)
    X_pretrain.append(torch.tensor(token_seq, dtype=torch.long))
    Y_pretrain.append(torch.tensor([1] + token_seq + [2], dtype=torch.long))

with open("token_to_id_pretrain.pkl", "wb") as f:
    pickle.dump(token_to_id_pretrain, f)
with open("id_to_token_pretrain.pkl", "wb") as f:
    pickle.dump(id_to_token_pretrain, f)

print("== Procesando entrenamiento supervisado ==")
X_train_files = []
Y_train_files = []

for file_name in os.listdir(TRAIN_DIR):
    index = file_name.split("_")[0]
    if file_name.startswith(index + "_1_pf_org"):
        X_train_files.append((int(index), os.path.join(TRAIN_DIR, file_name)))
    elif file_name.startswith(index + "_2_pf_rhm"):
        Y_train_files.append((int(index), os.path.join(TRAIN_DIR, file_name)))

X_train_files.sort(key=lambda x: x[0])
Y_train_files.sort(key=lambda x: x[0])

X_train_paths = [x[1] for x in X_train_files]
Y_train_paths = [y[1] for y in Y_train_files]

X_train_raw = [midi_to_chords_train(path) for path in X_train_paths]
Y_train_raw = [midi_to_chords_train(path) for path in Y_train_paths]

X_train_aug = []
Y_train_aug = []

for x_prog, y_prog in zip(X_train_raw, Y_train_raw):
    for shift in range(12):
        X_train_aug.append(transpose_progression(x_prog, shift))
        Y_train_aug.append(transpose_progression(y_prog, shift))

token_to_id_train, id_to_token_train = build_vocab(X_train_aug + Y_train_aug)

X_train = []
Y_train = []

for x_prog, y_prog in zip(X_train_aug, Y_train_aug):
    x_tokens = progression_to_token_seq(x_prog, token_to_id_train)
    y_tokens = progression_to_token_seq(y_prog, token_to_id_train)
    X_train.append(torch.tensor(x_tokens, dtype=torch.long))
    Y_train.append(torch.tensor([1] + y_tokens + [2], dtype=torch.long))

print(f"Total progresiones entrenamiento aumentadas: {len(X_train)}")

with open("token_to_id_train.pkl", "wb") as f:
    pickle.dump(token_to_id_train, f)
with open("id_to_token_train.pkl", "wb") as f:
    pickle.dump(id_to_token_train, f)


class PretrainDataset(Dataset):
    def __init__(self, X_data, Y_data):
        self.X = X_data
        self.Y = Y_data
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]
    @staticmethod
    def collate_fn(batch):
        X, Y = zip(*batch)
        return pad_sequence(X, batch_first=True, padding_value=0), pad_sequence(Y, batch_first=True, padding_value=0)

class TrainDataset(Dataset):
    def __init__(self, X_data, Y_data):
        self.X = X_data
        self.Y = Y_data
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]
    @staticmethod
    def collate_fn(batch):
        X, Y = zip(*batch)
        return pad_sequence(X, batch_first=True, padding_value=0), pad_sequence(Y, batch_first=True, padding_value=0)

pretrain_dataset = PretrainDataset(X_pretrain, Y_pretrain)
train_dataset = TrainDataset(X_train, Y_train)

pretrain_loader = DataLoader(pretrain_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=PretrainDataset.collate_fn)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=TrainDataset.collate_fn)

print(f"Pretrain dataset size: {len(pretrain_dataset)}")
print(f"Train dataset size: {len(train_dataset)}")

for xb, yb in pretrain_loader:
    print("Pretrain batch shapes - X:", xb.shape, "Y:", yb.shape)
    break

for xb, yb in train_loader:
    print("Train batch shapes - X:", xb.shape, "Y:", yb.shape)
    break

with open("id_to_token_pretrain.pkl", "rb") as f:
    id_to_token_pretrain = pickle.load(f)

ej_idx = 0
x_tokens, y_tokens = pretrain_dataset[ej_idx]


x_token_list = x_tokens.tolist()
y_token_list = y_tokens.tolist()

x_chords = [id_to_token_pretrain.get(tok, "?") for tok in x_token_list]
y_chords = [id_to_token_pretrain.get(tok, "?") for tok in y_token_list]

print("🔹 Secuencia X (entrada):")
print("Tokens:", x_token_list)
print("Acordes:", x_chords)

print("\n🔸 Secuencia Y (target):")
print("Tokens:", y_token_list)
print("Acordes:", y_chords)


== Procesando preentrenamiento ==
⚠️ Error leyendo PRE_TRAIN/.DS_Store: MThd not found. Probably not a MIDI file
Total progresiones preentrenamiento aumentadas: 1296
== Procesando entrenamiento supervisado ==
Total progresiones entrenamiento aumentadas: 600
Pretrain dataset size: 1296
Train dataset size: 600
Pretrain batch shapes - X: torch.Size([8, 16]) Y: torch.Size([8, 18])
Train batch shapes - X: torch.Size([8, 8]) Y: torch.Size([8, 10])
🔹 Secuencia X (entrada):
Tokens: [3, 4, 5, 6, 3, 7, 8, 7]
Acordes: ['0_5_7', '0_1_5_7', '0_2_5_7', '0_3_5_7', '0_5_7', '3_5_7_10', '1_5_7_8', '3_5_7_10']

🔸 Secuencia Y (target):
Tokens: [1, 3, 4, 5, 6, 3, 7, 8, 7, 2]
Acordes: ['<sos>', '0_5_7', '0_1_5_7', '0_2_5_7', '0_3_5_7', '0_5_7', '3_5_7_10', '1_5_7_8', '3_5_7_10', '<eos>']


In [ ]:

def chord_to_token(chord):
    return '_'.join(map(str, sorted(chord)))

def fallback_token_str(chord_str, token_to_id):
    matches = difflib.get_close_matches(chord_str, token_to_id.keys(), n=1)
    return token_to_id[matches[0]] if matches else token_to_id.get('<pad>', 0)

def progression_to_token_seq(prog, token_to_id):
    tokens = []
    for chord in prog:
        token = chord_to_token(chord)
        if token in token_to_id:
            tokens.append(token_to_id[token])
        else:
            tokens.append(fallback_token_str(token, token_to_id))
    return tokens

class ChordPairDataset(Dataset):
    def __init__(self, X_data, Y_data):
        self.X_data = X_data
        self.Y_data = Y_data

    def __len__(self):
        return len(self.X_data)

    def __getitem__(self, idx):
        return self.X_data[idx], self.Y_data[idx]

    @staticmethod
    def collate_fn(batch):
        X_batch, Y_batch = zip(*batch)
        X_padded = pad_sequence(X_batch, batch_first=True, padding_value=0)
        Y_padded = pad_sequence(Y_batch, batch_first=True, padding_value=0)
        return X_padded, Y_padded

X_data = []
Y_data = []

for x_prog, y_prog in zip(X_train_aug, Y_train_aug): 
    x_tokens = progression_to_token_seq(x_prog, token_to_id_train)
    y_tokens = [token_to_id_train['<sos>']] + progression_to_token_seq(y_prog, token_to_id_train) + [token_to_id_train['<eos>']]
    X_data.append(torch.tensor(x_tokens, dtype=torch.long))
    Y_data.append(torch.tensor(y_tokens, dtype=torch.long))

X_train_split, X_val_split, Y_train_split, Y_val_split = train_test_split(
    X_data, Y_data, test_size=0.2, random_state=42)

train_dataset = ChordPairDataset(X_train_split, Y_train_split)
val_dataset = ChordPairDataset(X_val_split, Y_val_split)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=ChordPairDataset.collate_fn)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, collate_fn=ChordPairDataset.collate_fn)

for X_batch, Y_batch in train_loader:
    print("X batch shape:", X_batch.shape)
    print("Y batch shape:", Y_batch.shape)
    break

print("Total batches de preentrenamiento:", len(pretrain_loader))
print("Total batches de train:", len(train_loader))

def token_seq_to_chords(token_seq, id_to_token):
    chords = []
    for idx in token_seq:
        token = id_to_token.get(idx, '<unk>')
        if token in ['<pad>', '<sos>', '<eos>']:
            continue
        chords.append(list(map(int, token.split('_'))))
    return chords


X batch shape: torch.Size([8, 7])
Y batch shape: torch.Size([8, 10])
Total batches de preentrenamiento: 162
Total batches de train: 60


In [ ]:
token_to_id = {'<pad>': 0, '<sos>': 1, '<eos>': 2}
id_to_token = {0: '<pad>', 1: '<sos>', 2: '<eos>'}

def chord_to_token(chord):
    if isinstance(chord, torch.Tensor):
        if chord.ndim == 0:
            chord = [chord.item()]
        else:
            chord = chord.tolist()
    return '_'.join(str(n) for n in sorted(chord))

def construir_vocabulario(datasets):
    for data in datasets:
        for prog in data:
            for chord in prog:
                token = chord_to_token(chord)
                if token not in token_to_id:
                    idx = len(token_to_id)
                    token_to_id[token] = idx
                    id_to_token[idx] = token

construir_vocabulario([X_train, Y_train, X_pretrain, Y_pretrain])
vocab_size = len(token_to_id)

print(f"✅ Vocabulario construido: {vocab_size} tokens")

class CVAE_LSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim, vocab_size, embedding_dim, num_layers=2, dropout=0.2):
        super(CVAE_LSTM, self).__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.latent_dim = latent_dim
        self.vocab_size = vocab_size

        # Encoder
        self.encoder_embedding = nn.Embedding(vocab_size, embedding_dim)
        self.encoder_lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout)
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

        # Decoder
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.decoder_lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout)
        self.latent_to_hidden = nn.Linear(latent_dim, hidden_dim * num_layers)
        self.output_layer = nn.Linear(hidden_dim, vocab_size)

    def encode(self, x):
        x = self.encoder_embedding(x)
        _, (h_n, _) = self.encoder_lstm(x)
        h_last = h_n[-1]
        mu = self.fc_mu(h_last)
        logvar = self.fc_logvar(h_last)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z, y_seq, teacher_forcing_ratio=0.4):
        batch_size, seq_len = y_seq.size()
        embedding = self.embedding

        hidden_state = torch.tanh(self.latent_to_hidden(z))
        hidden = hidden_state.view(self.decoder_lstm.num_layers, batch_size, self.hidden_dim)
        cell = torch.zeros_like(hidden).to(hidden.device)

        inputs = y_seq[:, 0]
        outputs = []

        for t in range(1, seq_len):
            input_embed = embedding(inputs).unsqueeze(1)
            output, (hidden, cell) = self.decoder_lstm(input_embed, (hidden, cell))
            output_logits = self.output_layer(output.squeeze(1))
            outputs.append(output_logits)

            teacher_force = (torch.rand(1).item() < teacher_forcing_ratio)
            top1 = output_logits.argmax(1)
            inputs = y_seq[:, t] if teacher_force else top1

        outputs = torch.stack(outputs, dim=1)
        return outputs


    def forward(self, x, y_seq=None, teacher_forcing_ratio=0.8):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        if y_seq is not None:
            y_hat = self.decode(z, y_seq, teacher_forcing_ratio)
            return y_hat, mu, logvar
        else:
            y_hat = self.generate(z, max_len=10)
            return y_hat, mu, logvar
    def generate(self, z, max_len=10, start_token_id=None, eos_token_id=None):
        batch_size = z.size(0)
        hidden_state = torch.tanh(self.latent_to_hidden(z))
        hidden = hidden_state.view(self.decoder_lstm.num_layers, batch_size, self.hidden_dim)
        cell = torch.zeros_like(hidden).to(hidden.device)

        inputs = torch.full((batch_size,), start_token_id, dtype=torch.long).to(z.device)
        generated_tokens = []

        for _ in range(max_len):
            input_embed = self.embedding(inputs).unsqueeze(1)
            output, (hidden, cell) = self.decoder_lstm(input_embed, (hidden, cell))
            output_logits = self.output_layer(output.squeeze(1))

            probs = torch.softmax(output_logits, dim=-1)
            inputs = torch.multinomial(probs, num_samples=1).squeeze(1)

            generated_tokens.append(inputs)

            if eos_token_id is not None:
                if (inputs == eos_token_id).all():
                    break

        generated_tokens = torch.stack(generated_tokens, dim=1)
        return generated_tokens

def detectar_escala(notas_midi):
    s = stream.Stream()
    for n in notas_midi:
        s.append(note.Note(n))
    k = s.analyze('key')
    escala = k.getScale().getPitches()
    escala_mod12 = sorted({p.pitchClass for p in escala})
    return escala_mod12

def tokens_to_root_notes(tokens):
    root_notes = []
    for token in tokens:
        chord = id_to_token[token]
        if chord in ['<sos>', '<eos>', '<pad>']:
            continue
        try:
            notas = [int(x) for x in chord.split('_')]
            root_note = min(notas)
            root_notes.append(root_note)
        except ValueError:
            continue
    return root_notes


def loss_coherencia_tonal_soft(y_hat_step, root_note, escala_mod12):
    probs = F.softmax(y_hat_step, dim=-1)
    fuera_escala = torch.zeros(probs.size(-1), device=probs.device)
    
    for idx in range(probs.size(-1)):
        chord = id_to_token[idx]
        if chord in ['<sos>', '<eos>', '<pad>']:
            fuera_escala[idx] = 0
        else:
            try:
                notas = [root_note + int(x) for x in chord.split('_')]
                count_fuera = sum(1 for n in notas if n % 12 not in escala_mod12)
                fuera_escala[idx] = count_fuera / len(notas)
            except:
                fuera_escala[idx] = 0
    loss = torch.sum(probs * fuera_escala)
    return loss

def loss_movimiento_armonico_soft(root_notes, device='cpu'):
    loss = torch.tensor(0., device=device)
    for i in range(1, len(root_notes)):
        salto = abs(root_notes[i] - root_notes[i-1])
        salto_tensor = torch.tensor(salto - 7.0, device=device)
        loss += torch.relu(salto_tensor)
    return loss / max(1, len(root_notes)-1)


def loss_tensiones_soft(y_hat_step):
    probs = F.softmax(y_hat_step, dim=-1)
    penalizacion = torch.zeros(probs.size(-1), device=probs.device)

    for idx in range(probs.size(-1)):
        chord = id_to_token[idx]
        if chord in ['<sos>', '<eos>', '<pad>']:
            penalizacion[idx] = 0
        else:
            try:
                num_notas = len(chord.split('_'))
                penalizacion[idx] = 1 if num_notas < 4 else 0
            except:
                penalizacion[idx] = 0

    loss = torch.sum(probs * penalizacion)
    return loss

def cvae_loss(y_hat, y, mu, logvar, beta=0.1, free_bits=1.0, pad_idx=0):
    y_target = y[:, 1:]
    recon_loss = F.cross_entropy(
        y_hat.reshape(-1, y_hat.size(-1)),
        y_target.reshape(-1),
        reduction='sum',
        ignore_index=pad_idx 
    )
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    kl_loss = torch.clamp(kl_loss, min=free_bits)
    return recon_loss + beta * kl_loss, recon_loss, kl_loss


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CVAE_LSTM(
    input_dim=32, 
    hidden_dim=64,
    embedding_dim=64,
    latent_dim=64,
    num_layers=1, 
    dropout=0.3,
    vocab_size=vocab_size,
).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def entrenar_cvae(model, dataloader, optimizer, device, beta=0.1, teacher_forcing_ratio=0.8, usar_music_loss=False, w_coherencia=2.0, w_tensiones=0.1, w_movimiento=0.3):
    model.train()
    total_loss, total_recon, total_kl, total_music = 0, 0, 0, 0

    for x, y in dataloader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()

        y_hat, mu, logvar = model(x, y, teacher_forcing_ratio)
        loss, recon, kl = cvae_loss(y_hat, y, mu, logvar, beta)

        if usar_music_loss:
            batch_music_loss = 0
            for b in range(y.size(0)):
                y_hat_seq = y_hat[b]
                y_seq = y[b]
                entrada = x[b].tolist()
                root_note = None
                for token_id in entrada:
                    token = id_to_token[token_id]
                    if token not in ['<sos>', '<pad>', '<eos>']:
                        try:
                            notas = [int(n) for n in token.split('_')]
                            root_note = min(notas)
                            break
                        except:
                            continue
                if root_note is None:
                    root_note = 0

                escala = detectar_escala([root_note + int(n) for n in id_to_token[entrada[1]].split('_')])
                for t in range(y_hat_seq.size(0)):
                    y_hat_step = y_hat_seq[t]
                    music1 = loss_coherencia_tonal_soft(y_hat_step, root_note, escala)
                    music2 = loss_tensiones_soft(y_hat_step)
                    batch_music_loss += w_coherencia * music1 + w_tensiones * music2

                root_pred = tokens_to_root_notes(y_hat[b].argmax(dim=1).tolist())
                batch_music_loss += w_movimiento * loss_movimiento_armonico_soft(root_pred, device=device)
            batch_music_loss = batch_music_loss / (y.size(0) * y_hat.size(1))
            loss += batch_music_loss
            total_music += batch_music_loss.item()

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_recon += recon.item()
        total_kl += kl.item()

    return total_loss, total_recon, total_kl, total_music


✅ Vocabulario construido: 456 tokens


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/nn/modules/rnn.py:83: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.3 and num_layers=1
  warnings.warn("dropout option adds dropout after all but last "


In [ ]:
from torch.utils.data import DataLoader, TensorDataset
from torch.nn.utils.rnn import pad_sequence

def preparar_dataset(progresiones_x, progresiones_y=None, pad_value=0):
    x_tensors = [torch.tensor(p, dtype=torch.long) for p in progresiones_x]
    x_padded = pad_sequence(x_tensors, batch_first=True, padding_value=pad_value)

    if progresiones_y is not None:
        y_tensors = [torch.tensor(p, dtype=torch.long) for p in progresiones_y]
        y_padded = pad_sequence(y_tensors, batch_first=True, padding_value=pad_value)
        return TensorDataset(x_padded, y_padded)
    else:
        return TensorDataset(x_padded, x_padded)


dataset_pre = preparar_dataset(X_pretrain)
loader_pre = DataLoader(dataset_pre, batch_size=8, shuffle=True)

print("== Fase 1: Preentrenamiento ==")
for epoch in range(600):
    loss, recon, kl, _ = entrenar_cvae(model, loader_pre, optimizer, device,
                                       teacher_forcing_ratio=1, usar_music_loss=False)
    print(f"[{epoch+1}] Loss: {loss:.2f} | Recon: {recon:.2f} | KL: {kl:.2f}")



== Fase 1: Preentrenamiento ==


/var/folders/j8/8v21hcxs2wg1v9dljnmt6sfr0000gn/T/ipykernel_22898/3423647319.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x_tensors = [torch.tensor(p, dtype=torch.long) for p in progresiones_x]


[1] Loss: 75437.22 | Recon: 75341.38 | KL: 958.46
[2] Loss: 69265.88 | Recon: 69170.60 | KL: 952.81
[3] Loss: 65409.33 | Recon: 65342.49 | KL: 668.33
[4] Loss: 62245.19 | Recon: 62198.42 | KL: 467.72
[5] Loss: 59722.99 | Recon: 59696.27 | KL: 267.24
[6] Loss: 57601.70 | Recon: 57580.85 | KL: 208.58
[7] Loss: 55787.86 | Recon: 55764.75 | KL: 231.14
[8] Loss: 54161.69 | Recon: 54137.74 | KL: 239.55
[9] Loss: 52722.80 | Recon: 52694.12 | KL: 286.82
[10] Loss: 51392.45 | Recon: 51357.65 | KL: 347.99
[11] Loss: 50180.02 | Recon: 50141.15 | KL: 388.72
[12] Loss: 49038.34 | Recon: 48988.96 | KL: 493.74
[13] Loss: 47945.85 | Recon: 47891.88 | KL: 539.65
[14] Loss: 46993.80 | Recon: 46927.25 | KL: 665.49
[15] Loss: 46057.51 | Recon: 45985.01 | KL: 724.95
[16] Loss: 45160.38 | Recon: 45077.73 | KL: 826.57
[17] Loss: 44315.65 | Recon: 44228.24 | KL: 874.09
[18] Loss: 43469.38 | Recon: 43375.38 | KL: 939.91
[19] Loss: 42714.31 | Recon: 42611.27 | KL: 1030.37
[20] Loss: 41967.18 | Recon: 41867.45 |

In [ ]:
def mostrar_reconstrucciones(model, dataset, id_to_token, device, token_to_id, num_muestras=5, max_len=8):
    model.eval()
    x_batch, _ = next(iter(DataLoader(dataset, batch_size=num_muestras, shuffle=True)))
    x_batch = x_batch.to(device)

    with torch.no_grad():
        mu, logvar = model.encode(x_batch)
        z = model.reparameterize(mu, logvar)
        generated = model.generate(
            z,
            max_len=max_len,
            start_token_id=token_to_id['<sos>'],
            eos_token_id=token_to_id['<eos>']
        )

    for i in range(num_muestras):
        entrada_tokens = x_batch[i].cpu().tolist()
        salida_tokens = generated[i].cpu().tolist()

        entrada_chords = [
            id_to_token[tok]
            for tok in entrada_tokens
            if tok not in [token_to_id['<pad>'], token_to_id['<eos>'], token_to_id['<sos>']]
        ]
        salida_chords = [
            id_to_token[tok]
            for tok in salida_tokens
            if tok not in [token_to_id['<pad>'], token_to_id['<eos>'], token_to_id['<sos>']]
        ]

        print(f"\n🔹 Entrada  ({i+1}): {' | '.join(entrada_chords)}")
        print(f"🔸 Reconstr ({i+1}): {' | '.join(salida_chords)}")

print("== Mostrando reconstrucciones del preentrenamiento ==")
mostrar_reconstrucciones(model, dataset_pre, id_to_token_pretrain, device, token_to_id_pretrain, num_muestras=5, max_len=8)



== Mostrando reconstrucciones del preentrenamiento ==

🔹 Entrada  (1): 0_4_5_7_9 | 0_5_6_8_10 | 0_1_3_5_8 | 1_5_6_8_10 | 2_5_7_9_10 | 0_2_4_9_10 | 2_4_5_7_9 | 2_5_7_9_10 | 1_5_6_8_10
🔸 Reconstr (1): 5_7_9_11 | 5_7_10_11 | 1_3_5_11 | 0_4_7_9 | 0_4_6_9 | 0_3_9_11 | 2_4_7_11 | 2_4_7_11

🔹 Entrada  (2): 4_8_11 | 2_5_8_11 | 1_4_6_9 | 3_6_9_11 | 3_6_8_11 | 1_5_8_11 | 1_4_6_9 | 3_6_9_11 | 1_4_8_10 | 1_4_7_9 | 3_6_8_11 | 1_4_7_10 | 1_4_6_9 | 0_3_5_9
🔸 Reconstr (2): 1_4_8_10 | 1_3_7_11 | 3_6_8_10_11 | 1_3_7_11 | 1_6_10 | 3_6_8_11 | 1_6_10 | 1_6_10

🔹 Entrada  (3): 0_3_7_8_10 | 3_6_8_10_11 | 2_5_7_10 | 3_4_6_8_10 | 0_3_5_7_8 | 2_5_7_10 | 0_3_7_8_10 | 0_1_5_8_10 | 1_3_5_7
🔸 Reconstr (3): 3_6_8_10_11 | 1_3_4_8_11 | 2_4_6_7_11 | 1_4_7_9_11 | 1_2_4_6_9 | 3_6_8_11 | 1_4_8_11 | 1_4_6_8_9

🔹 Entrada  (4): 3_6_11 | 3_6_9 | 4_8_11 | 4_7_11 | 3_6_11 | 1_5_8 | 4_8_11 | 4_6_8_11 | 1_4_6_10
🔸 Reconstr (4): 1_3_5_8_10 | 1_6_8_10 | 0_6_8_10 | 0_1_3_5_8 | 1_6_8_10 | 0_6_8_10 | 1_3_6_10_11 | 3_6_8_11

🔹 Entrada 

In [330]:
dataset_train = preparar_dataset(X_train, Y_train)
loader_train = DataLoader(dataset_train, batch_size=8, shuffle=True)

print("\n== Fase 2: Entrenamiento supervisado ==")
for epoch in range(100):
    loss, recon, kl, music = entrenar_cvae(model, loader_train, optimizer, device,
                                           teacher_forcing_ratio=0.9, usar_music_loss=True, w_coherencia=2.0, w_tensiones=0.1, w_movimiento=0.3)
    print(f"[{epoch+1}] Loss: {loss:.2f} | Recon: {recon:.2f} | KL: {kl:.2f} | Music: {music:.2f}")


/var/folders/j8/8v21hcxs2wg1v9dljnmt6sfr0000gn/T/ipykernel_22898/3423647319.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x_tensors = [torch.tensor(p, dtype=torch.long) for p in progresiones_x]
/var/folders/j8/8v21hcxs2wg1v9dljnmt6sfr0000gn/T/ipykernel_22898/3423647319.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_tensors = [torch.tensor(p, dtype=torch.long) for p in progresiones_y]



== Fase 2: Entrenamiento supervisado ==
[1] Loss: 5472.00 | Recon: 5076.91 | KL: 2814.97 | Music: 113.59
[2] Loss: 5693.54 | Recon: 5292.99 | KL: 2855.32 | Music: 115.02
[3] Loss: 5245.76 | Recon: 4839.92 | KL: 2931.13 | Music: 112.73
[4] Loss: 6072.39 | Recon: 5669.82 | KL: 2886.41 | Music: 113.94
[5] Loss: 5709.23 | Recon: 5302.01 | KL: 2927.08 | Music: 114.52
[6] Loss: 5946.70 | Recon: 5531.43 | KL: 2988.72 | Music: 116.40
[7] Loss: 5334.40 | Recon: 4920.17 | KL: 3010.32 | Music: 113.20
[8] Loss: 5510.67 | Recon: 5098.55 | KL: 2962.30 | Music: 115.89
[9] Loss: 5451.08 | Recon: 5043.24 | KL: 2944.34 | Music: 113.41
[10] Loss: 5303.38 | Recon: 4897.54 | KL: 2924.36 | Music: 113.41
[11] Loss: 5714.13 | Recon: 5308.44 | KL: 2925.30 | Music: 113.16
[12] Loss: 5713.57 | Recon: 5303.76 | KL: 2972.16 | Music: 112.59
[13] Loss: 5691.82 | Recon: 5276.47 | KL: 2991.27 | Music: 116.22
[14] Loss: 5299.14 | Recon: 4884.24 | KL: 3016.32 | Music: 113.27
[15] Loss: 5315.02 | Recon: 4908.70 | KL: 29

In [338]:
# guardar el modelo entrenado
torch.save(model.state_dict(), "cvae_lstm_model.pth")